# PROJET - IN304 
## InPoDa
### MEZOUER Amin et KHIDER Sarah




## Voici une explication des fonctions de traitement/analyse de données et de leur but :

1. **`charger_fichier`**: Cette fonction prend en paramètre le nom d'un fichier et renvoie le contenu du fichier en tant que chaîne de caractères.

2. **`nettoyer_fichier`**: Cette fonction prend en paramètre le nom d'un fichier, charge son contenu, le nettoie en supprimant les caractères non ASCII, puis enregistre le résultat dans un nouveau fichier appelé "atterrissage.json". Elle renvoie également le contenu nettoyé.

3. **`tweet_max_rt`**: Cette fonction prend en paramètre un fichier de tweets, extrait le nombre de retweets de chaque tweet, trouve le tweet avec le nombre maximum de retweets, et renvoie le texte de ce tweet.

4. **`extraire_nb_retweet`**: Cette fonction prend en paramètre un fichier de tweets et une clé, puis extrait les valeurs associées à cette clé (ici, le nombre de retweets).

5. **`extraire_hashtags`**: Cette fonction prend en paramètre un fichier de tweets, analyse chaque tweet pour extraire les hashtags, et renvoie une chaîne de caractères indiquant la présence ou l'absence de hashtags dans chaque tweet.

6. **`analyser_sentiment`**: Cette fonction prend en paramètre un fichier de tweets, analyse le sentiment de chaque tweet à l'aide de la bibliothèque TextBlob, et renvoie une chaîne de caractères indiquant si le sentiment est positif, négatif ou neutre.

7. **`top_k_hashtags`**: Cette fonction prend en paramètre un fichier de tweets et un entier k, extrait tous les hashtags, compte leur fréquence, puis renvoie les k hashtags les plus fréquents.

8. **`top_k_utilisateurs_mentionnes`**: Cette fonction prend en paramètre un fichier de tweets et un entier k, extrait tous les utilisateurs mentionnés, compte leur fréquence, puis renvoie les k utilisateurs les plus mentionnés.

9. **`nombre_publications_par_utilisateur`**: Cette fonction prend en paramètre un fichier de tweets, compte le nombre de publications par utilisateur en utilisant le champ "id", et renvoie le résultat.

10. **`tweets_mentions_utilisateur`**: Cette fonction prend en paramètre un fichier de tweets et un nom d'utilisateur, trouve tous les tweets mentionnant cet utilisateur, puis renvoie ces tweets.

11. **`utilisateurs_mentionnant_hashtag`**: Cette fonction prend en paramètre un fichier de tweets et un hashtag, trouve tous les utilisateurs mentionnant ce hashtag, puis renvoie ces utilisateurs.

12. **`nombre_publications_par_hashtag`**: Cette fonction prend en paramètre un fichier de tweets, compte le nombre de publications par hashtag en utilisant le champ "TweetText", et renvoie le résultat.

13. **`top_k_tweet_utilisateur`**:  (non finalisé) Cette fonction prend en paramètre un fichier de tweets et un entier k, extrait tous les utilisateurs, compte leur fréquence, puis renvoie les k utilisateurs avec le plus grand nombre de tweets.

14. **`utilisateurs_mentions_par_utilisateur`**:  (non finalisé) Cette fonction prend en paramètre un fichier de tweets et un nom d'utilisateur, trouve tous les utilisateurs mentionnés par cet utilisateur, puis renvoie ces utilisateurs.

15. **`detecter_topics`**: Cette fonction prend en paramètre le texte d'un tweet, détecte les thèmes associés en utilisant le dictionnaire Mot_clé, et renvoie une liste de thèmes détectés.

16. **`classement_topic`**:  Cette fonction prend en paramètre un fichier de tweets, analyse chaque tweet pour détecter les thèmes, compte le nombre de tweets par thème, puis renvoie un classement des thèmes en fonction du nombre de tweets.

17. **`lancer_interface`**:  Cette fonction initialise une interface Gradio avec des champs de texte et des menus déroulants pour permettre à l'utilisateur d'effectuer différentes opérations sur un fichier de tweets. Elle lance ensuite cette interface.





Enfin, la classe `Generale` crée une interface Gradio qui permet à l'utilisateur de choisir une opération parmi celles décrites ci-dessus, de spécifier un fichier et d'autres paramètres, puis d'afficher les résultats de l'opération sélectionnée. L'interface est ensuite lancée grâce à la méthode `lancer_interface`.


## Références

* https://www.gradio.app ( documentation pour l'interface )


* https://courspython.com/dictionnaire.html (aide pour la structure dictionnaire)


* https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html ( aide pour la structure dictionnaire )


* https://chat.openai.com (pour générer les mots clés des topics) 


* https://textblob.readthedocs.io/en/dev/quickstart.html ( documentation pour analyse de sentiment )


* https://youtu.be/H7JDoS4vLMU?si=VSE_yaO8lwsHH-sx ( aide pour l'interface)

In [1]:
import gradio as gr
import json
from collections import Counter
import re
import textblob
import datetime

class Generale:
    def __init__(self):
        #Définition des topics et des mots clés associés à ces topics 
        self.Mot_clé = {
            "Politics": ["Politics","Legislation", "Executive Order", "Constitution", "Democracy", "Bureaucracy", "Citizenship", "Diplomacy", 'Policy-making', "Public Administration", "Election", "Governance", "Integrity", "Sovereignty", "National Security", "Checks and Balances", "Foreign Relations", "Public Service", "Human Rights", "Budgetary Allocation"],
            "Music": ["Music","Melody", "Harmony", "Rhythm", "Lyrics", "Genre", "Beat", "Chorus", "Bridge", "Tempo", "Instrumentation", "Composition", "Performance", "Dynamics", "Arrangement", "Solo", "Duet", "Concert", "Recording", "Album"],
            "Sports": ["Sports","Football","sport", "Basketball", "Baseball", "Soccer", "Tennis", "Golf", "Cricket", "Hockey", "Swimming", "Athletics", "Cycling", "Boxing", "Volleyball", "Rugby", "Wrestling", "Martial Arts", "Gymnastics", "Skiing", "Snowboarding", "Surfing"],
            "Artificial Intelligence": ["Artificial Intelligence","Machine learning", "neural networks", "deep learning", "natural language processing", "computer vision", "robotics", "AI ethics", "automated reasoning", "expert systems", "chatbots", "data science", "pattern recognition", "intelligent agents", "cognitive computing"],
            "Food":["Food","Cuisine", "recipes", "gastronomy", "food culture", "culinary arts", "nutrition", "cooking techniques", "food blogging", "farm-to-table", "gourmet", "food photography", "taste testing", "food trends", "culinary innovation", "food sustainability", "food pairing"],
            "Medicine":["Medicine", "healthcare", "medical research", "biotechnology", "pharmaceuticals", "patient care", "surgery", "disease prevention", "public health", "epidemiology", "medical imaging", "telemedicine", "healthcare technology", "genetic medicine", "clinical trials", "healthcare policy", "medical ethics"],
            "Automobiles":["Automobiles", "cars", "automotive industry", "electric vehicles", "hybrid cars", "autonomous vehicles", "car technology", "automotive design", "fuel efficiency", "car manufacturing", "connected cars", "self-driving cars", "electric charging infrastructure", "automotive innovation", "car safety", "performance vehicles"],
            "Transportation":["Transportation", "public transit", "commuting", "mass transit", "bicycles", "scooters", "ride-sharing", "electric scooters", "trains", "buses", "aviation", "air travel", "ships", "maritime transport", "hyperloop", "future of transportation", "sustainable transportation", "smart cities", "mobility solutions"],
            "Education":["Education", "Learning", "Teaching", "Schools", "Students", "Teachers", "Curriculum", "Classrooms", "Online Learning", "E-Learning", "Educational Technology", "Distance Education", "Higher Education", "Elementary Education", "Secondary Education", "Educational Resources", "Digital Learning", "Pedagogy", "Academic Excellence", "STEM Education"],
            "Fashion":["Fashion", "Style", "Trends", "Clothing", "Apparel", "Designer", "Fashionista", "Runway", "Haute Couture", "Street Style", "Accessories", "Fashion Week", "Fashion Industry", "Fashion Design", "Fashion Blogging", "Fashion Influencer", "Fashion Photography", "Fashion Forward", "Wardrobe", "Fashion Statement"],
            "Music":["Music","Song", "Music", "Melody", "Lyrics", "Tune", "Harmony", "Rhythm", "Verse", "Chorus", "Bridge", "Composition", "Musical Genre", "Singer", "Artist", "Album", "Beat", "Instrumental", "Performance", "Recording", "Playlist"],
            "Environment":["Environment", "Sustainability", "Climate Change", "Green Living", "Renewable Energy", "Conservation", "Biodiversity", "Ecology", "Environmental Activism", "Carbon Footprint", "Eco-Friendly", "Zero Waste", "Pollution", "Natural Resources", "Global Warming", "Ocean Conservation", "Wildlife Protection", "Environmental Policy", "Green Technology", "Sustainable Development"] }


        #Etablissement de l'interface
        self.interface = gr.Interface(
            fn=self.toutes_fonctions, title="InPoDa",
            inputs=[
                gr.Textbox(label="Nom Du Fichier", placeholder="Entrez le nom du fichier ici..."),
                gr.Dropdown(["Charger Fichier", "Nettoyer Fichier", "Tweet Max RT", "Hashtags", "Analyse Sentiment", "Top K Hashtags", "Top K Utilisateurs Mentionnés", "Nombre de Publications par Utilisateur", "Tweets Mentionnant Utilisateur", "Utilisateurs Mentionnant Hashtag", "Nombre de Publications par Hashtag", "Classement Topic"], label="Choisissez L'Opération"),
                gr.Textbox(label="Hashtag/Utilisateur Pour L'Opération Utilisateurs Mentionnant Hashtag ou Tweets Mentions Utilisateur", placeholder="Entrez le hashtag ou le nom d'utilisateur ici...", type="text"),
                gr.Number(label="Nombre De Hashtags À Afficher Pour L'Opération Top K Hashtags ou Top K Utilisateurs Mentionnés", value=1)],
            outputs=gr.Textbox(label="Résultat", lines=6990))
        
        
    #Regroupement de toutes les fonctions de traitement/d'analyse de données dans une seule fonction pour faciliter la manipulation pour l'interface
    def toutes_fonctions(self, fichier, operation, hashtag_utilisateur, k):
        k = int(k) if k and int(k) > 0 else 1
        if operation == "Charger Fichier":
            return self.charger_fichier(fichier)
        elif operation == "Nettoyer Fichier":
            return self.nettoyer_fichier(fichier)
        elif operation == "Tweet Max RT":
            return self.tweet_max_rt(fichier)
        elif operation == "Hashtags":
            return self.extraire_hashtags(fichier)
        elif operation == "Analyse Sentiment":
            return self.analyser_sentiment(fichier)
        elif operation == "Top K Hashtags":
            return self.top_k_hashtags(fichier, k)
        elif operation == "Top K Utilisateurs Mentionnés":
            return self.top_k_utilisateurs_mentionnes(fichier, k)
        elif operation == "Nombre de Publications par Utilisateur":
            return self.nombre_publications_par_utilisateur(fichier)
        elif operation == "Nombre de Publications par Hashtag":
            return self.nombre_publications_par_hashtag(fichier, )
        elif operation == "Tweets Mentionnant Utilisateur":
            return self.tweets_mentions_utilisateur(fichier, hashtag_utilisateur)
        elif operation == "Utilisateurs Mentionnant Hashtag":
            return self.utilisateurs_mentionnant_hashtag(fichier, hashtag_utilisateur)
        elif operation == "Classement Topic":
            return self.classement_topic(fichier)
        


    def charger_fichier(self, fichier_a_charger):
        with open(fichier_a_charger, 'r', encoding='utf-8') as fichier:     #Ouvre le fichier en mode lecture ('r') avec l'encodage UTF-8
            contenu = fichier.read()                                        #Lit le contenu du fichier et le stocke dans la variable 'contenu'
        return contenu                                                      #Renvoie le contenu du fichier 



    def nettoyer_fichier(self, fichier_a_nettoyer):
        contenu_original = self.charger_fichier(fichier_a_nettoyer)                      #Charge le contenu du fichier en utilisant la fonction charger_fichier
        contenu_nettoye = contenu_original.encode('ascii', 'ignore').decode('ascii')     #Supprime les caractères pas ASCII du fichier
        with open('atterrissage.json', 'w') as fichier_nettoye:                          #Ouvre le fichier nommé 'atterrissage.json' en mode écriture ('w')
            fichier_nettoye.write(contenu_nettoye)                                       #Écrit le contenu nettoyé dans le fichier 'atterrissage.json'
        return contenu_nettoye                                                           #Renvoie le contenu nettoyé
    


    def tweet_max_rt(self, fichier):                                    
        t = self.extraire_nb_retweet(fichier, "RetweetCount")           #Appelle la fonction extraire_nb_retweet pour avoir la liste des valeurs de RetweetCount (le nombres de retweets)
        t1 = [int(e) for e in t]                                        #Convertit les valeurs en entiers car ce sont des chaines de caractères dans le fichier json fourni
        max_rt = max(t1)                                                #Trouve la valeur maximale parmi le nombre retweets
        dico_max_rt = None                                              #Initialise un dictionnaire vide qui contiendra le tweet avec le maximum de retweets
        with open(fichier, 'r') as f:                                   #Ouvre le fichier en mode lecture ("r")
            for ligne in f:                                             #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                          #Charge la ligne 
                if str(max_rt) in str(objet_json):                      #Cherche  la valeur maximale de retweets dans le dictionnaire 
                    dico_max_rt = objet_json                            #Affecte le dictionnaire avec le maximum de retweets à la variable
        return dico_max_rt["TweetText"]                                 #Renvoie le texte du tweet qui a le maximum de retweets



    def extraire_nb_retweet(self, fichier, cle):                        
        valeurs = []                                                    #Initialise une liste pour stocker les nombres de retweets
        with open(fichier, 'r') as f:                                   #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                             #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                          #Charge la ligne JSON 
                if cle in objet_json:                                   #Vérifie si la clé que l'on cherche est présente dans le dictionnaire du tweet
                    valeurs.append(objet_json[cle])                     #Ajoute la valeur de la clé à la liste des valeurs si elle est présente
        return valeurs                                                  #Renvoie la liste des nombes de retweets



    def extraire_hashtags(self, fichier):                               
        hashtags_present = []                                           #Initialise une liste pour stocker les hashtags présents dans les tweets
        with open(fichier, 'r') as f:                                   #Ouvre le fichier en mode lecture("r")
            for i, ligne in enumerate(f, 1):                            #Parcourt chaque ligne du fichier avec un compteur en commencant à 1
                objet_json = json.loads(ligne)                          #Charge la ligne JSON
                tweet_text = objet_json.get("TweetText", "")            #Récupère le texte du tweet à partir du dictionnaire JSON
                hashtags = re.findall(r'#\w+', tweet_text)              #Utilise une expression régulière pour trouver tous les hashtags dans le texte du tweet
                if hashtags:
                    hashtags_present.append(f"Hashtags présents dans le tweet {i} : {', '.join(hashtags)}")  #Si des hashtags sont trouvés on les ajoute à la liste en indiquant le numéro de ligne du tweet
                else:
                    hashtags_present.append(f"Le tweet {i} ne contient pas de hashtags")  #Sinon, ajoute un message indiquant l'absence de hashtags dans le tweet
        return "\n".join(hashtags_present)                              



    def analyser_sentiment(self, fichier):                          
        sentiments = []                                                 #Initialise une liste pour stocker les résultats d'analyse de sentiment
        with open(fichier, 'r') as f:                                   #Ouvre le fichier en mode lecture("r")
            for i, ligne in enumerate(f, 1):                            #Parcourt chaque ligne du fichier avec un compteur qui commence à 1
                objet_json = json.loads(ligne)                          #Charge la ligne JSON 
                tweet_text = objet_json.get("TweetText", "")            #Récupère le texte du tweet à partir du dictionnaire JSON
                analysis = textblob.TextBlob(tweet_text)                #Crée un objet TextBlob pour effectuer l'analyse de sentiment sur le texte du tweet grace a cette bibliotheque 
                sentiment = "Positif" if analysis.sentiment.polarity > 0 else "Négatif" if analysis.sentiment.polarity < 0 else "Neutre"  #Détermine le sentiment en fonction de la polarité de l'analyse
                sentiments.append(f"Le sentiment du tweet {i} est : {sentiment}")  #Ajoute le résultat à la liste avec un message indiquant le numéro de ligne du tweet
        return "\n".join(sentiments)                                 




    def top_k_hashtags(self, fichier, k):                         
        hashtags = []                                              #Initialise une liste pour stocker tous les hashtags du fichier
        with open(fichier, 'r') as f:                              #Ouvre le fichier en mode lecture ("r")
            for ligne in f:                                        #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                     #Charge la ligne JSON 
                tweet_text = objet_json.get("TweetText", "")       #Récupère le texte du tweet à partir du dictionnaire JSON
                hashtags.extend(re.findall(r'#\w+', tweet_text))   #Utilise une expression régulière pour extraire tous les hashtags du texte du tweet et les ajoute à la liste
        counter = Counter(hashtags)                                #Utilise Counter pour compter le nombre d'occurrences de chaque hashtag
        top_k = counter.most_common(k)                             #Sélectionne les k hashtags les plus fréquents
        if not top_k:                                              #Vérifie s'il n'y a pas de hashtags
            return "Aucun hashtag trouvé dans le fichier."         #Renvoie un message s'il n'y a pas de hashtags
        result = [f"Top {k} des hashtags les plus fréquents :"]    #Initialise une liste pour stocker les résultats
        for i, (tag, count) in enumerate(top_k, 1):                #Parcourt les k hashtags les plus fréquents avec un compteur qui démarre à 1
            result.append(f"{i}. {tag} ({count} apparitions)")     #Ajoute chaque hashtag avec sa ligne et son nombre d'apparitions à la liste des résultats
        return "\n".join(result)                                  




    def top_k_utilisateurs_mentionnes(self, fichier, k):             
        utilisateurs = []                                              #Initialise une liste pour stocker tous les utilisateurs mentionnés du fichier
        with open(fichier, 'r') as f:                                  #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                            #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                         #Charge la ligne JSON
                tweet_text = objet_json.get("TweetText", "")           #Récupère le texte du tweet à partir du dictionnaire JSON
                mentions = re.findall(r'@\w+', tweet_text)             #Utilise une expression régulière pour extraire tous les utilisateurs mentionnés du texte du tweet
                if mentions:                                           #Vérifie s'il y a des utilisateurs mentionnés
                    utilisateurs.extend(mentions)                      #Ajoute tous les utilisateurs mentionnés à la liste
        counter = Counter(utilisateurs)                                #Utilise Counter pour compter le nombre d'occurrences de chaque utilisateur mentionné
        top_k = counter.most_common(k)                                 #Sélectionne les k utilisateurs mentionnés les plus fréquents
        if not top_k:                                                  #Vérifie s'il n'y a pas d'utilisateurs mentionnés
            return "Aucun utilisateur mentionné trouvé dans le fichier."  #Renvoie un message s'il n'y a pas d'utilisateurs mentionnés
        result = [f"Top {k} des utilisateurs les plus mentionnés :"]   #Initialise une liste pour stocker les résultats
        for i, (utilisateur, count) in enumerate(top_k, 1):            #Parcourt les k utilisateurs mentionnés les plus fréquents avec un compteur demarrant à 1
            result.append(f"{i}. {utilisateur} ({count} mentions)")    #Ajoute chaque utilisateur mentionné avec son rang et son nombre de mentions à la liste des résultats
        return "\n".join(result)                                       




    def nombre_publications_par_utilisateur(self, fichier):            
        utilisateurs_publications = Counter()                          #Initialise un objet Counter pour compter le nombre de publications par utilisateur
        with open(fichier, 'r') as f:                                  #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                            #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                         #Charge la ligne JSON
                id = objet_json.get("id", "")                          #Récupère l'identifiant de l'utilisateur à partir du dictionnaire JSON
                utilisateurs_publications[id] += 1                     #Incrémente le nombre de publications pour l'utilisateur actuel
        result = [f"Nombre de publications par utilisateur :"]         #Initialise une liste pour stocker les résultats
        for i, (utilisateur, count) in enumerate(utilisateurs_publications.items(), 1):  #Parcourt chaque utilisateur avec son nombre de publications, avec un compteur qui commence à 1
            result.append(f"{utilisateur} : {count} publication(s)")   #Ajoute chaque utilisateur avec son nombre de publications à la liste des résultats
        return "\n".join(result)                                       




    def tweets_mentions_utilisateur(self, fichier, utilisateur):      
        tweets_mentionnant = []                                       #Initialise une liste pour stocker les tweets mentionnant l'utilisateur
        with open(fichier, 'r') as f:                                 #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                           #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                        #Charge la ligne JSON 
                tweet_text = objet_json.get("TweetText", "")          #Récupère le texte du tweet à partir du dictionnaire JSON
                mentions = re.findall(fr'@{utilisateur}\b', tweet_text, flags=re.IGNORECASE)  #Recherche les mentions de l'utilisateur dans le texte du tweet
                if mentions:                                          #Si des mentions sont trouvées
                    tweets_mentionnant.append(tweet_text)             #Ajoute le texte du tweet à la liste des tweets mentionnant l'utilisateur
        if not tweets_mentionnant:                                    #Si la liste des tweets mentionnant est vide
            return f"Aucun tweet mentionnant {utilisateur} trouvé dans le fichier."  #Renvoie un message indiquant qu'aucun tweet mentionnant l'utilisateur n'a été trouvé
        result = [f"Tweets mentionnant {utilisateur} :"]              #Initialise une liste pour stocker les résultats
        for i, tweet in enumerate(tweets_mentionnant, 1):             #Parcourt chaque tweet mentionnant l'utilisateur, avec un compteur qui commence à 1
            result.append(f"{i}. {tweet}")                            #Ajoute chaque tweet avec son numéro de ligne à la liste des résultats
        return "\n".join(result)                                      #Renvoie la liste des résultats sous forme de chaîne de caractères




    def utilisateurs_mentionnant_hashtag(self, fichier, hashtag):  
        utilisateurs_mentionnant = set()                            #Initialise un ensemble pour stocker les utilisateurs mentionnant le hashtag
        with open(fichier, 'r') as f:                               #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                         #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                      #Charge la ligne JSON 
                tweet_text = objet_json.get("TweetText", "")        #Récupère le texte du tweet à partir du dictionnaire JSON
                mentions = re.findall(fr'@\w+', tweet_text)         #Recherche les mentions d'utilisateurs dans le texte du tweet
                hashtags = re.findall(fr'#{re.escape(hashtag)}\b', tweet_text, flags=re.IGNORECASE)  #Recherche le hashtag dans le texte du tweet
                if hashtags:                                        #Si le hashtag est trouvé
                    utilisateurs_mentionnant.update(mentions)       #Ajoute les utilisateurs mentionnés 
        if not utilisateurs_mentionnant:                            #Si l'ensemble d'utilisateurs mentionnant est vide
            return f"Aucun utilisateur mentionnant #{hashtag} trouvé dans le fichier."  #Renvoie un message indiquant qu'aucun utilisateur mentionnant le hashtag n'a été trouvé
        result = [f"Utilisateurs mentionnant #{hashtag} :"]         #Initialise une liste pour stocker les résultats
        for i, utilisateur in enumerate(utilisateurs_mentionnant, 1):  #Parcourt chaque utilisateur mentionnant le hashtag, avec un compteur commençant à 1
            result.append(f"{i}. {utilisateur}")                   #Ajoute chaque utilisateur avec son numéro de ligne à la liste des résultats
        return "\n".join(result)                                   #Renvoie la liste des résultats sous forme de chaîne de caractères

    
    
    
    def nombre_publications_par_hashtag(self, fichier):           
        hashtags_publications = Counter()                           #Initialise un objet Counter pour compter les occurrences de chaque hashtag
        with open(fichier, 'r') as f:                               #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                         #Parcourt chaque ligne du fichier
                tweet_text = json.loads(ligne).get("TweetText", "") #Charge la ligne JSON et récupère le texte du tweet
                hashtags = re.findall(r'#\w+', tweet_text)          #Recherche tous les hashtags dans le texte du tweet
                hashtags_publications.update(hashtags)              #Met à jour le compteur avec les hashtags trouvés
        result = [f"Nombre de publications par hashtag :"]          #Initialise une liste pour stocker les résultats
        for hashtag, count in hashtags_publications.items():        #Parcourt chaque hashtag avec son compteur associé
            result.append(f"{hashtag} : {count} publication(s)")    #Ajoute chaque hashtag et son nombre de publications à la liste des résultats
        return "\n".join(result)                                    

    
    
    
    def detecter_topics(self, tweet_text):                        
        detected_topics = []                                       #Initialise une liste pour stocker les thèmes détectés
        for topic, keywords in self.Mot_clé.items():               #Parcourt chaque thème et les mots-clés associés dans le dictionnaire Mot_clé
            for keyword in keywords:                               #Parcourt chaque mot-clé associé à un thème
                if keyword.lower() in tweet_text.lower():          #Vérifie si le mot-clé (en minuscules) est présent dans le texte du tweet (en minuscules)
                    detected_topics.append(topic)                  #Ajoute le thème détecté à la liste
                    break                                          #Sort de la boucle interne car un seul thème doit être ajouté par tweet
        return detected_topics                                     #Renvoie la liste des thèmes détectés




    def classement_topic(self, fichier):                          
        topics_counts = {topic: 0 for topic in self.Mot_clé}        #Initialise un dictionnaire pour stocker le nombre de tweets par thème
        with open(fichier, 'r') as f:                               #Ouvre le fichier en mode lecture("r")
            for ligne in f:                                         #Parcourt chaque ligne du fichier
                objet_json = json.loads(ligne)                      #Charge la ligne JSON 
                tweet_text = objet_json.get("TweetText", "")        #Extrait le texte du tweet de l'objet JSON
                detected_topics = self.detecter_topics(tweet_text)  #Détecte les thèmes dans le texte du tweet
                for topic in detected_topics:                       #Parcourt chaque thème détecté
                    topics_counts[topic] += 1                       #Incrémente le compteur pour ce thème
        sorted_topics = sorted(topics_counts.items(), key=lambda x: x[1], reverse=True)   #Trie les thèmes par nombre de tweets décroissant
        return "\n".join([f"{topic}: {count} tweets" for topic, count in sorted_topics])  #Renvoie une liste formatée avec le classement des thèmes

    
    

    def lancer_interface(self):                        
        self.interface.launch()                        #Lance l'interface utilisateur définie dans l'objet Generale

interface_finale = Generale()                      #Crée une instance de la classe Generale appelée interface_finale
interface_finale.lancer_interface()                #Appelle la méthode lancer_interface pour lancer l'interface utilisateur


/opt/miniconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## Opération DataFrame

Ces fonctions permettent de stocker touts les élements du tweet dans un dictionnaire. On parours chaque ligne du fichier et grace a la ligne dictionnaire = json.loads(ligne) on convertis la ligne du fichier en dictionnaire qu'on stock dans le dictionnaire nommé data. On transforme ensuite notre dictionnaire en data frame panda grace a la ligne df = pd.DataFrame(data) avant de retourner notre data frame.

In [2]:
import json
import pandas as pd

def stockage_tweet(fichier):                    
    data = []                                   #Initialise une liste vide pour stocker les données
    with open(fichier, 'r') as f:               #Ouvre le fichier en mode lecture
        lignes = f.readlines()                  #Lit toutes les lignes du fichier
        for ligne in lignes:                    #Parcourt chaque ligne
            dictionnaire = json.loads(ligne)    #Charge le contenu de la ligne en tant que dictionnaire JSON
            data.append(dictionnaire)           #Ajoute le dictionnaire à la liste de données
    df = pd.DataFrame(data)                     #Crée un DataFrame à partir des données
    return df                                   #Renvoie le DataFrame

print(stockage_tweet("atterissage.json"))      


def recent(fichier):                            
    data = []                                   #Initialise une liste vide pour stocker les données
    with open(fichier, 'r') as f:               #Ouvre le fichier en mode lecture
        lignes = f.readlines()                  #Lit toutes les lignes du fichier
        for ligne in lignes:                    #Parcourt chaque ligne
            dictionnaire = json.loads(ligne)    #Charge le contenu de la ligne en tant que dictionnaire JSON
            data.append(dictionnaire)           #Ajoute le dictionnaire à la liste de données
    df = pd.DataFrame(data)                     #Crée un DataFrame à partir des données
    df['CreatedAt'] = pd.to_datetime(df['CreatedAt'], format='%Y-%m-%dT%H:%M:%SZ')  #Convertit la colonne 'CreatedAt' en format datetime
    df = df.sort_values(by='CreatedAt', ascending=False)  #Trie le DataFrame par 'CreatedAt' en ordre décroissant
    return df                                   #Renvoie le DataFrame

print(recent("atterissage.json"))              



                       id        AuthorLocation             CreatedAt  \
0     1415291904850153474                        2021-07-14T12:47:39Z   
1     1415291947560828933   Mysore  and  BERLIN  2021-07-14T12:47:49Z   
2     1415291877897605120                        2021-07-14T12:47:33Z   
3     1415291886860967940                        2021-07-14T12:47:35Z   
4     1415291968700264450              Internet  2021-07-14T12:47:54Z   
...                   ...                   ...                   ...   
1703  1421408743770791938    Liverpool, England  2021-07-31T09:53:47Z   
1704  1421424066305605634                        2021-07-31T10:54:40Z   
1705  1421423882427371521             127.0.0.1  2021-07-31T10:53:57Z   
1706  1421423971858149377  Singapore, Singapore  2021-07-31T10:54:18Z   
1707  1421423964845395969   Torremolinos, Espaa  2021-07-31T10:54:16Z   

     RetweetCount TweetLanguage  \
0               0            en   
1               2            en   
2             246 

## Fonctions que l'on a pas réussi a faire fonctionner

In [3]:

def utilisateurs_mentions_par_utilisateur(self, fichier, utilisateur):
    utilisateurs_mentions = Counter()

    with open(fichier, 'r') as f:
        for ligne in f:
            # Recherche des mentions de l'utilisateur spécifié dans le texte du tweet
            mentions = re.findall(fr'@{utilisateur}\b', json.loads(ligne).get("TweetText", ""), flags=re.IGNORECASE)
            utilisateurs_mentions.update(mentions)
    if not utilisateurs_mentions:
        return f"Aucun utilisateur mentionné par {utilisateur} trouvé dans le fichier."
    result = [f"Utilisateurs mentionnés par {utilisateur} :"]
    for i, (mention, count) in enumerate(utilisateurs_mentions.items(), 1):
        result.append(f"{i}. {mention} ({count} mentions)")
    return "\n".join(result)



def top_k_tweet_utilisateur(self, fichier, k):
    utilisateurs_tweet = Counter()
    with open(fichier, 'r') as f:
        for ligne in f:
            objet_json = json.loads(ligne)
            utilisateur = objet_json.get("UserName", "")
            utilisateurs_tweet[utilisateur] += 1
    top_k = utilisateurs_tweet.most_common(k)
    if not top_k:
        return "Aucun utilisateur trouvé dans le fichier."
    result = [f"Top {k} des utilisateurs avec le plus de tweets :"]
    for i, (utilisateur, count) in enumerate(top_k, 1):
        result.append(f"{i}. {utilisateur} ({count} tweets)")

    return "\n".join(result)
